# 🧪 실험 3 (v3): EDA 기반 개선판 — 속도 + 과적합 + 데이터 품질

---

## 📊 v3에서 추가된 점 (v2 → v3)

실제 데이터 EDA에서 발견한 3가지 문제를 추가로 반영했습니다.

| # | 발견한 문제 | v3 개선 | 효과 |
|---|------------|---------|------|
| ① | 16:9 원본을 224 정사각으로 강제 리사이즈 → **가로 1.78배 찌그러짐** | **letterbox 패딩 리사이즈** | 병변 형태 보존 (A4·A6에 특히 유리) |
| ② | `class_weight`가 질환 기준 → 질환은 균형이라 **효과 거의 없음** | **`sample_weight`로 종(반려묘) 기준 가중** | 소수 클래스(반려묘) 성능 실질 향상 |
| ③ | 현미경(CYT) 이미지가 일반카메라(IMG)와 섞임 → 해상도·밝기 분포 상이 | **IMG만 필터링** | 학습 분포 일관성 확보 |

### v2에서 이미 적용된 것 (유지)
- ⚡ 속도: tf.data prefetch, 증강 경량화(rotation 15), batch 32
- 📉 과적합: Dropout 0.5, EarlyStopping patience 3, ReduceLR factor 0.3

> 💾 저장 경로: `exp3_weighted_v3.h5` (v2 결과 보존)

> ⚠️ **EDA 핵심 사실**: 반려묘는 A2·A4·A6·A7 4개 클래스에만 존재하고 **A1·A3·A5에는 0장**입니다. 따라서 종별 성능 비교는 이 4개 클래스에서만 유효합니다.

---

|------|------|------|------|
| 🐢 속도 | 데이터 파이프라인 | ImageDataGenerator | **+ tf.data prefetch** | GPU 유휴 제거 (에폭 20분→5~8분) |
| 🐢 속도 | `rotation_range` | 30 | **15** | CPU 전처리 부하 감소 |
| 🐢 속도 | `BATCH_SIZE` | 64 | **32** | 소수 클래스 등장 빈도↑ |
| 📉 과적합 | `Dropout` | 0.3 | **0.5** | 정확도 하락 직접 차단 |
| 📉 과적합 | `ReduceLROnPlateau factor` | 0.5 | **0.3** | 과적합 구간 LR 빠른 감소 |
| 📉 과적합 | `EarlyStopping patience` | 5 | **3** | 하락 시작 시 즉시 종료 |

> 💡 `LEARNING_RATE`는 **1e-4 그대로 유지**합니다. 정확도가 오르다 떨어지는 건 LR이 낮아서가 아니라 과적합이 원인이므로, LR을 올리면 오히려 악화됩니다.


---

## 이 모델이 하는 일

실험 1, 2와 동일하게 **반려동물 피부 사진 → 7가지 질환 분류**를 수행합니다.

```
📸 반려동물 피부 사진 입력
        ↓
    [EfficientNetB0 + 커스텀 분류층 + 종별 가중치]
        ↓
    A4 농포/여드름 : 92.4%  ← 예측 결과 (반려묘도 잘 맞춤!)
```

## 실험 1, 2와 무엇이 다른가?

| 항목 | 실험 1 | 실험 2 | 실험 3 (이번) |
|------|--------|--------|-------------|
| **모델** | 직접 설계 CNN | EfficientNetB0 | EfficientNetB0 |
| **증강** | ❌ 없음 | ✅ 적용 | ✅ 적용 |
| **Class Weight** | ❌ 없음 | ❌ 없음 | ✅ **적용** |
| **핵심 질문** | 분류 가능한가? | 성능이 올라가는가? | **소수 클래스(반려묘) 성능이 올라가는가?** |

## 왜 Class Weight가 필요한가?

현재 데이터에 **종별 불균형**이 존재합니다:

```
반려견(D): 31,010장 (88.6%) ← 압도적으로 많음
반려묘(C):  3,990장 (11.4%) ← 매우 적음
```

Class Weight 없이 학습하면 모델이 **반려견 데이터에 편향**되어, 반려묘 이미지를 잘 분류하지 못할 수 있습니다.

## Class Weight 동작 원리

```
반려견(D) 가중치: 0.563  → 흔한 데이터니까 loss 기여도를 줄임
반려묘(C) 가중치: 4.443  → 희귀한 데이터니까 loss 기여도를 4.4배 높임
```

모델이 반려묘 이미지를 틀리면 **4.4배 큰 벌점(loss)**을 받으므로, 반려묘도 잘 맞추도록 학습됩니다.

## 기대 효과

- 전체 정확도는 실험 2와 비슷하거나 약간 변동
- **반려묘 클래스의 Recall/F1이 향상** (소수 클래스 성능 개선)
- 더 공정한 모델 → **실제 서비스 배포에 적합한 최종 후보**

---


## 0. 라이브러리 임포트 및 설정

실험 2와 동일합니다. 추가로 **class_weight 계산을 위한 sklearn** 유틸리티를 임포트합니다.


In [14]:
!pip install tensorflow

^C


  Using cached tensorflow-2.21.0-cp312-cp312-win_amd64.whl.metadata (4.5 kB)
  Using cached absl_py-2.4.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached astunparse-1.6.3-py2.py3-none-any.whl.metadata (4.4 kB)
  Using cached flatbuffers-25.12.19-py2.py3-none-any.whl.metadata (1.0 kB)
  Using cached gast-0.7.0-py3-none-any.whl.metadata (1.5 kB)
  Using cached google_pasta-0.2.0-py3-none-any.whl.metadata (814 bytes)
  Using cached libclang-18.1.1-py2.py3-none-win_amd64.whl.metadata (5.3 kB)
  Using cached opt_einsum-3.4.0-py3-none-any.whl.metadata (6.3 kB)
  Using cached protobuf-7.35.1-cp310-abi3-win_amd64.whl.metadata (595 bytes)
  Using cached requests-2.34.2-py3-none-any.whl.metadata (4.8 kB)
  Using cached termcolor-3.3.0-py3-none-any.whl.metadata (6.5 kB)
  Using cached wrapt-2.2.1-cp312-cp312-win_amd64.whl.metadata (7.6 kB)
  Using cached grpcio-1.81.1-cp312-cp312-win_amd64.whl.metadata (3.8 kB)
  Using cached keras-3.14.1-py3-none-any.whl.metadata (6.3 kB)
  Using cached h5py-3

ERROR: Could not install packages due to an OSError: [WinError 32] 다른 프로세스가 파일을 사용 중이기 때문에 프로세스가 액세스 할 수 없습니다: 'D:\\SAG\\venv\\Lib\\site-packages\\certifi\\__main__.py'
Check the permissions.



In [3]:
import torch

print(f"PyTorch 버전: {torch.__version__}")
print(f"CUDA 사용 가능: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU 개수: {torch.cuda.device_count()}")
    print(f"현재 GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA 버전: {torch.version.cuda}")
    device = torch.device("cuda")
else:
    print("GPU를 찾을 수 없어 CPU로 실행됩니다")
    device = torch.device("cpu")

print(f"사용 디바이스: {device}")

AttributeError: partially initialized module 'torch' has no attribute 'types' (most likely due to a circular import)

In [ ]:
import os
import numpy as np
import cv2  # v3: letterbox 리사이즈용
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

import tensorflow as tf
from tensorflow import keras

from keras.applications import EfficientNetB0
from keras.models import Model
from keras.layers import (
    Dense,
    Dropout,
    GlobalAveragePooling2D,
    Input
)
from keras.callbacks import (
    EarlyStopping,
    ModelCheckpoint,
    ReduceLROnPlateau
)
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight  # ⭐ 실험 3 추가: 클래스 가중치 자동 계산
import seaborn as sns

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

print(f"TensorFlow 버전: {tf.__version__}")
print(f"GPU 사용 가능: {len(tf.config.list_physical_devices('GPU')) > 0}")


ModuleNotFoundError: No module named 'tensorflow'

  Using cached tensorflow-2.21.0-cp312-cp312-win_amd64.whl.metadata (4.5 kB)
  Using cached absl_py-2.4.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached astunparse-1.6.3-py2.py3-none-any.whl.metadata (4.4 kB)
  Using cached gast-0.7.0-py3-none-any.whl.metadata (1.5 kB)
  Using cached google_pasta-0.2.0-py3-none-any.whl.metadata (814 bytes)
  Using cached requests-2.34.2-py3-none-any.whl.metadata (4.8 kB)
  Using cached grpcio-1.81.1-cp312-cp312-win_amd64.whl.metadata (3.8 kB)
  Using cached keras-3.14.1-py3-none-any.whl.metadata (6.3 kB)
  Using cached h5py-3.14.0-cp312-cp312-win_amd64.whl.metadata (2.7 kB)
  Using cached charset_normalizer-3.4.7-cp312-cp312-win_amd64.whl.metadata (41 kB)
  Using cached certifi-2026.5.20-py3-none-any.whl.metadata (2.5 kB)
  Using cached rich-15.0.0-py3-none-any.whl.metadata (18 kB)
  Using cached markdown_it_py-4.2.0-py3-none-any.whl.metadata (7.4 kB)
Using cached tensorflow-2.21.0-cp312-cp312-win_amd64.whl (350.9 MB)
Using cached grpcio-1.81.1-cp31

## 1. 경로 설정 및 데이터 로딩

실험 1, 2와 **동일한 데이터, 동일한 분할**을 사용합니다.  
실험 간 공정한 비교를 위해 데이터는 절대 바꾸지 않습니다.


In [ ]:
ROOT      = Path('..')
PROCESSED = ROOT / 'data' / 'processed'
FIG_DIR   = ROOT / 'outputs' / 'figures'
MODEL_DIR = ROOT / 'models'

FIG_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# CSV 로딩
df = pd.read_csv(PROCESSED / 'dataset_cleaned.csv')
target_col = 'img_path'

# ── ③ v3 개선: 현미경(CYT) 이미지 제외, 일반카메라(IMG)만 사용 ──
# CYT는 2048x1644·밝기 188로 일반카메라(1920x1080·밝기 135)와 분포가 완전히 다름
# 파일명 접두사로 구분: IMG_* (일반카메라) vs CYT_* (현미경)
before = len(df)
df = df[df['img_file'].str.startswith('IMG')].copy()
removed = before - len(df)
print(f"③ CYT(현미경) 제외: {removed:,}장 제거 → {len(df):,}장 유지\n")

df_train = df[df['split'] == 'train'].copy()
df_val   = df[df['split'] == 'val'].copy()
df_test  = df[df['split'] == 'test'].copy()

print(f"✅ 데이터 로드 완료 (IMG 일반카메라만)")
print(f"  train : {len(df_train):,}장")
print(f"  val   : {len(df_val):,}장")
print(f"  test  : {len(df_test):,}장")

# 종별 분포 확인 (EDA 기반 — 반려묘는 A2/A4/A6/A7에만 존재)
print(f"\n  [종별 분포]")
for sp in ['D', 'C']:
    label = '반려견' if sp == 'D' else '반려묘'
    cnt = len(df_train[df_train['species'] == sp])
    print(f"    {sp}({label}): {cnt:,}장")

## 2. 하이퍼파라미터 설정

실험 2와 동일합니다. Class Weight는 별도 단계에서 계산합니다.


In [ ]:
IMG_SIZE      = (224, 224)   # EfficientNetB0 ImageNet 사전학습 크기와 동일 (변경 금지)
BATCH_SIZE    = 32           # ⚡ v2: 64→32 (소수 클래스 반려묘가 배치마다 더 자주 등장 → class weight 효과↑, 메모리 여유↑)
EPOCHS        = 15           # ⚡ v2: 20→15 상한 (EarlyStopping이 보통 7~10에폭에서 종료)
NUM_CLASSES   = 7
LEARNING_RATE = 0.0001       # 🔒 v2: 1e-4 유지 (과적합이 원인이므로 LR 인상 금지)

print(f"이미지 크기  : {IMG_SIZE}")
print(f"배치 크기    : {BATCH_SIZE}  (v2: 64→32)")
print(f"최대 에폭    : {EPOCHS}  (v2: 20→15)")
print(f"클래스 수    : {NUM_CLASSES}")
print(f"학습률       : {LEARNING_RATE}  (유지)")

## 3. 가중치 계산 (v3: 종 기준 sample_weight)

> 🔄 **v3 변경**: 아래 표는 원본(v2) 설명입니다. v3에서는 질환 class_weight 대신 **종(species) 기준 sample_weight**를 학습에 사용합니다 (질환은 균형이라 효과가 없었기 때문).

### (원본 설명) Class Weight 계산 ⭐ (실험 3 핵심)

이 단계가 **실험 2와의 유일한 차이점**입니다.

### 실험 3에서 적용하는 가중치

| 가중치 | 기준 | 적용 방식 | 목적 |
|--------|------|----------|------|
| **class_weight** | 질환별 (A1~A7) | `model.fit(class_weight=...)` | 질환 클래스 불균형 보정 ✅ |
| species_weight | 종별 (반려견/반려묘) | 참고용 (학습 미적용) | 불균형 규모 파악 |

> 💡 종별 불균형(`species_weight`)은 데이터 현황 파악을 위해 계산하되, 학습에는 적용하지 않습니다.  
> 종별 불균형까지 보정하려면 `sample_weight`로 결합 가중치를 적용할 수 있으나, 실험 3에서는 질환별 `class_weight`만 사용합니다.

### sample_weight vs class_weight 차이

- `class_weight`: **클래스 단위** 가중치. A1이 적으면 A1 전체에 높은 가중치
- `sample_weight`: **샘플(이미지) 단위** 가중치. 같은 A1이라도 반려묘면 가중치가 더 높음

In [ ]:
# ── ② v3 개선: 종(species) 가중치를 sample_weight로 학습에 실제 적용 ──
# EDA 발견: 질환(lesion)은 거의 균형 → lesion 기준 class_weight는 효과 미미
# 진짜 불균형은 종(반려견 88.6% vs 반려묘 11.4%)이므로 종 기준으로 가중

species_classes = np.array(sorted(df_train['species'].unique()))
species_weights = compute_class_weight(
    class_weight='balanced',
    classes=species_classes,
    y=df_train['species']
)
species_weight_map = dict(zip(species_classes, species_weights))  # {'C': ~3.x, 'D': ~0.5x}

print("=== 종별(species) 가중치 (v3: 학습에 실제 적용) ===")
for k, v in species_weight_map.items():
    label = '반려묘' if k == 'C' else '반려견'
    count = len(df_train[df_train['species'] == k])
    print(f"  {k} ({label}): {v:.3f}  (데이터 {count:,}장)")
print("\n  → 이 가중치로 각 샘플의 sample_weight를 만들어 model.fit에 전달합니다.")

In [ ]:
# ── 3-2. 질환별(lesion) 클래스 가중치 계산 ──
# Generator의 class_indices 순서에 의존하지 않도록 이름→가중치 맵으로 저장
# class_weight_dict는 Generator 생성 후 class_indices를 참조하여 확정 (Section 4)

lesion_classes = sorted(df_train['lesion'].unique())
lesion_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.array(lesion_classes),
    y=df_train['lesion']
)
lesion_weight_map = dict(zip(lesion_classes, lesion_weights))  # {클래스명: 가중치}

print("=== 질환별(lesion) 클래스 가중치 ===")
for cls, w in lesion_weight_map.items():
    count = len(df_train[df_train['lesion'] == cls])
    print(f"  {cls} → 가중치 {w:.3f}  (데이터 {count:,}장)")

=== 질환별(lesion) 클래스 가중치 ===
  A1 → 가중치 1.000  (데이터 3,499장)
  A2 → 가중치 1.000  (데이터 3,499장)
  A3 → 가중치 1.000  (데이터 3,498장)
  A4 → 가중치 1.001  (데이터 3,497장)
  A5 → 가중치 1.000  (데이터 3,500장)
  A6 → 가중치 1.000  (데이터 3,499장)
  A7 → 가중치 1.000  (데이터 3,500장)


## 4. 데이터 Generator 생성 (증강 적용)

실험 2와 **동일한 증강 설정**입니다.  
Class Weight는 Generator가 아니라 **model.fit()에서 적용**합니다.


In [ ]:
# ── ① v3: tf.data 기반 letterbox 파이프라인 (ImageDataGenerator 대체) ──
# tf.image.resize_with_pad = TF 내장 letterbox. 종횡비 보존하며 224로 맞추고 부족분은 패딩.
# ImageDataGenerator의 강제 정사각 리사이즈 문제를 근본 해결 + GPU 가속 가능.

IMG_H, IMG_W = 224, 224

def decode_and_letterbox(path):
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize_with_pad(img, IMG_H, IMG_W)   # ① 종횡비 보존 letterbox
    img = tf.cast(img, tf.float32) / 255.0               # 정규화
    return img

def augment(img):
    # v2 증강 경량화 유지: 좌우반전 + 약한 회전/밝기. (피부 병변 도메인에 맞게 과하지 않게)
    img = tf.image.random_flip_left_right(img)
    img = tf.image.random_brightness(img, max_delta=0.15)        # ≈ brightness_range [0.85,1.15]
    img = tf.image.random_contrast(img, 0.9, 1.1)
    img = tf.clip_by_value(img, 0.0, 1.0)
    return img

print("✅ tf.data letterbox 파이프라인 정의 완료")
print("   ① tf.image.resize_with_pad로 16:9 종횡비 보존 (병변 왜곡 방지)")

In [ ]:
# 라벨 인코딩: lesion 문자열 → 정수 인덱스 → one-hot
lesion_to_idx = {cls: i for i, cls in enumerate(lesion_classes)}  # lesion_classes는 Section 3에서 정의
idx_to_lesion = {i: cls for cls, i in lesion_to_idx.items()}
print("클래스 인덱스:", lesion_to_idx)

def df_to_arrays(d):
    paths   = d[target_col].values
    labels  = np.array([lesion_to_idx[l] for l in d['lesion'].values], dtype=np.int32)
    weights = np.array([species_weight_map[s] for s in d['species'].values], dtype=np.float32)  # ② 종 가중치
    return paths, labels, weights

train_paths, train_labels, train_weights = df_to_arrays(df_train)
val_paths,   val_labels,   _             = df_to_arrays(df_val)
test_paths,  test_labels,  _             = df_to_arrays(df_test)

print(f"\n✅ 배열 변환 완료")
print(f"  train: {len(train_paths):,} / val: {len(val_paths):,} / test: {len(test_paths):,}")
print(f"  ② 종 sample_weight — 반려묘: {species_weight_map['C']:.3f}, 반려견: {species_weight_map['D']:.3f}")

## 4-1. tf.data 파이프라인 구성 (letterbox + 증강 + 종 sample_weight + prefetch)

ImageDataGenerator를 tf.data로 완전히 대체합니다. 한 파이프라인에서 4가지를 처리:
- ① **letterbox**: `tf.image.resize_with_pad`로 16:9 종횡비 보존
- ② **종 sample_weight**: 반려묘 샘플에 ~3배 가중 (질환 아닌 종 기준)
- ⚡ **prefetch**: GPU 유휴 제거 (에폭 시간 단축)
- 증강: train에만 적용

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE
BATCH_SIZE = 32   # v2 유지

# ── Train: (img, label, sample_weight) 3-튜플, 증강 O ──
def make_train_map(path, label, weight):
    img = decode_and_letterbox(path)
    img = augment(img)                               # train만 증강
    label_oh = tf.one_hot(label, NUM_CLASSES)
    return img, label_oh, weight                     # ② sample_weight 포함

train_ds = (
    tf.data.Dataset.from_tensor_slices((train_paths, train_labels, train_weights))
    .shuffle(len(train_paths), seed=42, reshuffle_each_iteration=True)
    .map(make_train_map, num_parallel_calls=AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)                              # ⚡ GPU 유휴 제거
)

# ── Val/Test: (img, label), 증강 X ──
def make_eval_map(path, label):
    img = decode_and_letterbox(path)
    return img, tf.one_hot(label, NUM_CLASSES)

val_ds = (
    tf.data.Dataset.from_tensor_slices((val_paths, val_labels))
    .map(make_eval_map, num_parallel_calls=AUTOTUNE)
    .batch(BATCH_SIZE).prefetch(AUTOTUNE)
)
test_ds = (
    tf.data.Dataset.from_tensor_slices((test_paths, test_labels))
    .map(make_eval_map, num_parallel_calls=AUTOTUNE)
    .batch(BATCH_SIZE).prefetch(AUTOTUNE)
)

steps_per_epoch  = int(np.ceil(len(train_paths) / BATCH_SIZE))
validation_steps = int(np.ceil(len(val_paths) / BATCH_SIZE))

print("✅ tf.data 파이프라인 구성 완료")
print(f"  train_ds: (img, label, sample_weight) — letterbox+증강+종가중+prefetch")
print(f"  steps_per_epoch={steps_per_epoch}, validation_steps={validation_steps}")

## 5. EfficientNetB0 + 커스텀 분류층 정의

실험 2와 **완전히 동일한 모델 구조**입니다.  
차이점은 모델 구조가 아니라 **학습 시 class_weight 적용 여부**입니다.

```
입력 (224×224×3)
    ↓
EfficientNetB0 (ImageNet, freeze)
    ↓
GlobalAveragePooling2D → 1280
    ↓
Dense(256, relu) → Dropout(0.3)
    ↓
Dense(7, softmax) → [A1~A7 확률]
```

In [ ]:
# EfficientNetB0 사전학습 모델 로드
base_model = EfficientNetB0(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)

# 사전학습 가중치 고정
base_model.trainable = False

# 커스텀 분류층 연결
inputs = Input(shape=(224, 224, 3))
x = base_model(inputs, training=False)
x = GlobalAveragePooling2D()(x)
x = Dense(256, activation='relu')(x)
x = Dropout(0.5)(x)          # 📉 v2: 0.3→0.5 (과적합 직접 차단 — 정확도 하락의 주원인 해결)
outputs = Dense(NUM_CLASSES, activation='softmax')(x)

model = Model(inputs=inputs, outputs=outputs)

print(f"✅ 모델 구성 완료 (v2: Dropout 0.3→0.5)")
print(f"  총 레이어 수: {len(model.layers)}")

## 6. 모델 컴파일

실험 2와 동일합니다. Class Weight는 compile이 아니라 **fit()에서 적용**합니다.


In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ efficientnetb0 (Functional)     │ (None, 7, 7, 1280)     │     4,049,571 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │       327,936 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 7)              │         1,799 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,379,306 (16.71 MB)

 Trainable params: 329,735 (1.26 MB)

 Non-trainable params: 4,049,571 (15.45 MB)

## 7. 콜백(Callbacks) 설정

저장 경로만 **`exp3_weighted.h5`**로 변경합니다.


In [ ]:
callbacks = [
    EarlyStopping(
        monitor='val_loss',
        patience=3,                  # 📉 v2: 5→3 (val_loss 하락 시작 시 즉시 종료 → 시간 절약)
        restore_best_weights=True,
        verbose=1
    ),

    ModelCheckpoint(
        filepath=str(MODEL_DIR / 'exp3_weighted_v3.h5'),   # 💾 v2: 경로 분리 (원본 미덮어씀)
        monitor='val_loss',
        save_best_only=True,
        verbose=1
    ),

    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.3,                  # 📉 v2: 0.5→0.3 (과적합 구간에서 LR을 더 공격적으로 감소)
        patience=2,                  # 📉 v2: 3→2 (LR 감소를 더 빠르게 트리거)
        min_lr=1e-7,
        verbose=1
    )
]

print("✅ 콜백 설정 완료 (v2: patience↓, factor↓)")
print(f"  모델 저장 경로: {MODEL_DIR / 'exp3_weighted_v3.h5'}")

## 8. 모델 학습 ⭐ (v3: 종 sample_weight 적용)

v3에서는 `class_weight`(질환 기준)를 **`sample_weight`(종 기준)**으로 대체했습니다.

**왜 바뀌었나?** EDA 결과 질환(A1~A7)은 거의 균형이라 질환 기준 가중치는 효과가 없었습니다.
진짜 불균형은 종(반려견 88.6% vs 반려묘 11.4%)이므로, 반려묘 샘플에 직접 더 큰 가중을 부여합니다.

```python
# 실험 3 v2 (질환 기준 — 효과 미미)
model.fit(train_ds, ..., class_weight=class_weight_dict)

# 실험 3 v3 (종 기준 sample_weight — train_ds에 (x,y,w)로 내장) ⭐
model.fit(train_ds, ...)   # train_ds가 종 가중치를 함께 흘려보냄
```

- 반려묘 이미지를 틀리면 → 약 3배 큰 벌점(loss)
- 반려견 이미지를 틀리면 → 약 0.5배 벌점
- → 데이터가 적은 반려묘(소수 종)도 잘 맞추도록 학습


In [ ]:
print("=" * 55)
print("🚀 실험 3 (v3): EDA 기반 개선판 학습 시작")
print("=" * 55)
print(f"\n[v3 적용 요약]")
print(f"  ① letterbox(resize_with_pad) — 16:9 종횡비 보존")
print(f"  ② 종 sample_weight — 반려묘 {species_weight_map['C']:.2f} / 반려견 {species_weight_map['D']:.2f}")
print(f"  ③ CYT 현미경 제외 (IMG만)")
print()

# train_ds가 (x, y, sample_weight) 3-튜플을 내보내므로 class_weight 불필요
history = model.fit(
    train_ds,
    epochs=EPOCHS,
    validation_data=val_ds,
    callbacks=callbacks,
    verbose=1
)

print("\n✅ 학습 완료!")
print(f"   실제 학습 에폭 수: {len(history.history['loss'])}회")

## 9. 학습 곡선 시각화

실험 2와 비교하여 **Val Loss/Accuracy가 어떻게 달라졌는지** 확인합니다.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history.history['loss'], label='Train Loss', linewidth=2)
axes[0].plot(history.history['val_loss'], label='Val Loss', linewidth=2)
axes[0].set_title('Loss 곡선', fontsize=14)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend(fontsize=12)
axes[0].grid(True)

axes[1].plot(history.history['accuracy'], label='Train Accuracy', linewidth=2)
axes[1].plot(history.history['val_accuracy'], label='Val Accuracy', linewidth=2)
axes[1].set_title('Accuracy 곡선', fontsize=14)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend(fontsize=12)
axes[1].grid(True)

plt.suptitle('실험 3: Transfer Learning + 증강 + Class Weight 학습 곡선', fontsize=16, fontweight='bold')
plt.tight_layout()

plt.savefig(FIG_DIR / 'exp3_learning_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"📊 학습 곡선 저장 → {FIG_DIR / 'exp3_learning_curve.png'}")


## 10. 테스트 데이터 평가


In [ ]:
print("=" * 50)
print("📝 테스트 데이터 평가")
print("=" * 50)

test_loss, test_acc = model.evaluate(test_ds, verbose=1)

print(f"\n테스트 Loss    : {test_loss:.4f}")
print(f"테스트 Accuracy: {test_acc:.4f} ({test_acc*100:.1f}%)")


## 11. 상세 평가 — Classification Report

실험 2 대비 **반려묘 관련 클래스의 Recall/F1이 올랐는지** 확인하는 것이 핵심입니다.


In [ ]:
# v3: tf.data는 .classes 속성이 없으므로 test_labels를 직접 사용
y_pred_proba = model.predict(test_ds)
y_pred = np.argmax(y_pred_proba, axis=1)
y_true = test_labels   # df_to_arrays에서 만든 정수 라벨 (test 순서 그대로, shuffle 없음)

class_names = [idx_to_lesion[i] for i in range(NUM_CLASSES)]
LESION_MAP = {
    'A1': 'A1 구진/플라크',
    'A2': 'A2 비듬/각질',
    'A3': 'A3 태선화/색소',
    'A4': 'A4 농포/여드름',
    'A5': 'A5 미란/궤양',
    'A6': 'A6 결절/종괴',
    'A7': 'A7 무증상(정상)'
}
class_labels = [LESION_MAP[c] for c in class_names]

print("=== Classification Report ===")
print(classification_report(y_true, y_pred, target_names=class_labels))

### 11-1. Confusion Matrix (혼동 행렬)

실험 2 대비 **대각선 값이 더 고르게 분포**하는지 확인하세요.  
특정 클래스만 잘 맞추는 게 아니라, 모든 클래스를 공평하게 맞추는 것이 목표입니다.


In [ ]:
cm = confusion_matrix(y_true, y_pred)

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_labels, yticklabels=class_labels, ax=ax)
ax.set_xlabel('예측 (Predicted)', fontsize=12)
ax.set_ylabel('실제 (Actual)', fontsize=12)
ax.set_title('실험 3: Transfer Learning + 증강 + Class Weight — Confusion Matrix', fontsize=14)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()

plt.savefig(FIG_DIR / 'exp3_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"📊 혼동 행렬 저장 → {FIG_DIR / 'exp3_confusion_matrix.png'}")


## 12. 종별(반려견/반려묘) 성능 분석 ⭐ (실험 3 전용)

Class Weight의 효과를 확인하기 위해, **반려견과 반려묘 각각의 정확도**를 측정합니다.  
실험 2 대비 **반려묘 정확도가 올랐는지**가 핵심 비교 포인트입니다.


In [ ]:
# ⚠️ v3 EDA 주의: 반려묘는 A2/A4/A6/A7 4개 클래스에만 존재 (A1/A3/A5는 0장)
#    → 반려묘 정확도는 사실상 이 4개 클래스에 대한 측정값임

# test_ds는 shuffle하지 않으므로 df_test 순서 == y_pred 순서
test_species = df_test['species'].values

dog_mask = (test_species == 'D')
cat_mask = (test_species == 'C')
dog_acc = np.mean(y_pred[dog_mask] == y_true[dog_mask])
cat_acc = np.mean(y_pred[cat_mask] == y_true[cat_mask]) if cat_mask.sum() > 0 else float('nan')

print("=" * 50)
print("📊 종별 정확도 분석 (v3)")
print("=" * 50)
print(f"  반려견(D) 정확도: {dog_acc:.4f} ({dog_acc*100:.1f}%) — {dog_mask.sum():,}장")
print(f"  반려묘(C) 정확도: {cat_acc:.4f} ({cat_acc*100:.1f}%) — {cat_mask.sum():,}장")
print(f"  전체     정확도: {test_acc:.4f} ({test_acc*100:.1f}%)")
print()
print("  ⚠️ 반려묘는 A2/A4/A6/A7 4개 클래스만 존재 (A1/A3/A5는 0장)")
print("  → v2(질환 class_weight) 대비 반려묘 정확도가 올랐는지가 핵심 비교 포인트")

## 13. 실험 3 결과 요약


In [ ]:
print("=" * 60)
print("📋 실험 3: Transfer Learning + 증강 + Class Weight 결과 요약")
print("=" * 60)
print(f"모델 구조       : EfficientNetB0 (freeze) + Dense 분류층")
print(f"데이터 증강     : 적용 (회전, 반전, 밝기, 줌, 이동)")
print(f"Class Weight    : ✅ 적용 (질환별 가중치)")
print(f"학습률          : {LEARNING_RATE}")
print(f"학습 에폭       : {len(history.history['loss'])}회")
print(f"최종 Train Loss : {history.history['loss'][-1]:.4f}")
print(f"최종 Train Acc  : {history.history['accuracy'][-1]:.4f}")
print(f"최종 Val Loss   : {history.history['val_loss'][-1]:.4f}")
print(f"최종 Val Acc    : {history.history['val_accuracy'][-1]:.4f}")
print(f"테스트 Loss     : {test_loss:.4f}")
print(f"테스트 Accuracy : {test_acc:.4f} ({test_acc*100:.1f}%)")
print(f"반려견 정확도   : {dog_acc:.4f} ({dog_acc*100:.1f}%)")
print(f"반려묘 정확도   : {cat_acc:.4f} ({cat_acc*100:.1f}%)")
print(f"모델 저장 경로  : models/exp3_weighted_v3.h5")
print("=" * 60)
print()
print("→ 다음: 실험 1, 2, 3 성능 비교 후 최종 모델 선정 → Streamlit 앱 탑재")
